# Unit 10B — The conditional 1D U-Net

**Your result:** the full denoiser network used by Diffusion Policy, assembled from the Unit 10A blocks. Fed the weights of the pinned LeRobot network it gives the same outputs, for the small test network and for the prepared checkpoint. You then change its architecture under a fixed budget and compare.

Three short exercises (about 10–20 minutes each) and a larger experiment (30–60 minutes). These times are estimates, not measurements. Full answers are in [the reference notebook](10b-unet-reference.ipynb).

**Sources.** [Diffusion Policy](https://diffusion-policy.cs.columbia.edu/) (temporal U-Net for actions), the U-Net idea in [DDPM](https://arxiv.org/abs/2006.11239) (appendix B), and the pinned `DiffusionConditionalUnet1d` in `lerobot/policies/diffusion/modeling_diffusion.py` (around line 654; revision `e0d50211ef236143ae867228662b7dfaba554f02`). The pinned model has **no EMA** in the current trainer; the weights you load are the raw trained weights.

**Where this unit sits.** Units 9 and 10A are available as package versions. Your functions here (`unet_channel_plan`, `unet_forward`, the weight converter) run inside the real network through its hooks.

In [ ]:
# Supplied: workspace, offline caches, imports
from pathlib import Path
import os
import sys


def find_root():
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository tree.")


WORKSPACE = find_root()
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))

# Set cache paths and offline mode before any data or model library is imported.
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
CONFIGS, DATA = WORKSPACE / "configs", WORKSPACE / ".local-data"
import time
import torch
from torch import nn
import matplotlib.pyplot as plt

import vla_lab.teaching.diffusion as ref  # package reference: used by checks only
from vla_lab.teaching.diffusion import (  # earlier units, as package versions
    ConditionalUnet1D, DDPMSchedule, DiffusionDenoiser, epsilon_loss_fn, film_residual_forward,
    flat_state_condition, forward_noise, load_prepared_diffusion, load_state_action_windows,
    LeRobotDenoiser, sinusoidal_timestep_embedding, take_batch, unet_from_lerobot, verify_unet_parity,
)
from vla_lab.teaching.act_model import open_libero_dataset
from vla_lab.teaching.training import train_loop
from lerobot.configs import FeatureType, PolicyFeature
from lerobot.policies.diffusion import DiffusionConfig
from lerobot.policies.diffusion.modeling_diffusion import DiffusionConditionalUnet1d

torch.manual_seed(0)
print("PyTorch:", torch.__version__)
print("Workspace:", WORKSPACE)

In [ ]:
# Supplied: a helper that the checks use
def close(actual, expected, message, atol=1e-5):
    """Compare two tensors and say what failed."""
    assert actual.shape == expected.shape, f"{message}: shape {tuple(actual.shape)}, expected {tuple(expected.shape)}"
    worst = (actual.detach() - expected.detach()).abs().max().item()
    assert worst <= atol, f"{message}: largest difference {worst:.2e} is above {atol:.0e}"

## Background: a U-Net over time

The input is an action window `(B, 16, 7)`. The U-Net treats the 16 steps as a 1D signal with 7 channels.

```
encoder:   blocks (7→d0)(d0→d0)  ─skip→  ↓2   blocks (d0→d1)(d1→d1) ─skip→  ↓2   blocks (d1→d2)(d2→d2) ─skip→
middle:    two blocks at width d2
decoder:   [x ‖ skip] (2·d2→d1)(d1→d1) ↑2    [x ‖ skip] (2·d1→d0)(d0→d0) ↑2     head: conv → conv(→7)
```

- Going *down*, the sequence is halved by a stride-2 convolution (all stages except the last) and the channel width grows.
- Each stage keeps a **skip feature**. Going *up*, the newest skip is concatenated on the channel axis, so the decoder blocks take twice the channels.
- Every block is conditioned on `cond = [timestep MLP output, global observation condition]`.
- The decoder has one stage fewer than the encoder (`n − 1` for `n` encoder stages). It upsamples after *every* decoder stage, including the last, with a transposed convolution. The sequence length falls by `2^(n-1)` in the encoder and rises by `2^(n-1)` in the decoder, so it returns to 16.

**Supplied:** `Conv1dBlock`, the residual block with your Unit 10A function, the module class `ConditionalUnet1D` (it builds itself from your plan function and runs your forward function), and the data reader. **You write** the channel plan, the forward pass with skip connections, and the weight converter.

In [ ]:
# Supplied: a small real pinned U-Net and algorithm-check inputs
IMAGES = ("observation.images.image", "observation.images.image2")


def pinned_unet(down_dims=(16, 32, 64), cond_dim=6, seed=4):
    """A *real* pinned DiffusionConditionalUnet1d with random weights (7 actions, horizon 16)."""
    torch.manual_seed(seed)
    config = DiffusionConfig(
        input_features={
            IMAGES[0]: PolicyFeature(FeatureType.VISUAL, (3, 64, 64)),
            IMAGES[1]: PolicyFeature(FeatureType.VISUAL, (3, 64, 64)),
            "observation.state": PolicyFeature(FeatureType.STATE, (8,)),
        },
        output_features={"action": PolicyFeature(FeatureType.ACTION, (7,))},
        n_obs_steps=2, horizon=16, n_action_steps=8, drop_n_last_frames=7, down_dims=tuple(down_dims),
        diffusion_step_embed_dim=32, pretrained_backbone_weights=None, push_to_hub=False, device="cpu",
    )
    unet = DiffusionConditionalUnet1d(config, global_cond_dim=cond_dim)
    for parameter in unet.parameters():          # distinct biases and scales make wrong wiring visible
        if parameter.dim() == 1:
            nn.init.normal_(parameter, std=0.1)
    return unet.eval()


PINNED = pinned_unet()
X, T_STEPS, COND = torch.randn(3, 16, 7), torch.tensor([0, 10, 99]), torch.randn(3, 6)    # algorithm-check inputs (random numbers)
print("pinned U-Net parameters:", sum(p.numel() for p in PINNED.parameters()))

## Exercise 10B.1 — The channel plan (about 15 min)

**Purpose.** Decide which block takes how many channels, where the sequence is downsampled, and what the decoder receives after the skip connection is concatenated. A wrong plan builds a network that cannot load the real weights.

**Contract (shapes, units, masks).**

- `unet_channel_plan(action_dim, down_dims)` returns a dict with the keys `"down"`, `"mid"`, `"up"`.
- Let `pairs = [(action_dim, down_dims[0]), (down_dims[0], down_dims[1]), …, (down_dims[-2], down_dims[-1])]`.
- `"down"`: one entry per pair, `(in_channels, out_channels, downsample)`. Each stage has two residual blocks (`in → out`, `out → out`). `downsample` is `True` for every stage **except the last**.
- `"mid"`: `[(c, c), (c, c)]` with `c = down_dims[-1]`.
- `"up"`: one entry per pair **in reverse order, leaving out the first pair** `(action_dim, down_dims[0])`. For a reversed pair `(width_out, width_in)` (that is `(d_{k-1}, d_k)` read as `(out, in)`) the entry is `(2 * width_in, width_out, True)`: the first block takes the decoder state plus the concatenated skip, hence `2 * width_in`. The pinned network upsamples after every `"up"` stage, so `upsample` is `True` for all of them.

**Method and source.** Read the constructor of the pinned `DiffusionConditionalUnet1d`: the lists `in_out`, `down_modules`, `mid_modules`, `up_modules`. The check builds the real network and compares every channel number and every stride.

**Pointers.** First build `pairs`. Then make `down` from `pairs` with a flag for the last one. Make `up` from `reversed(pairs[1:])`. Print your plan for `(8, 16, 32)`-style inputs and compare with the picture above.

<details><summary>Detailed hint</summary>

```python
pairs = [(action_dim, down_dims[0]), *zip(down_dims[:-1], down_dims[1:])]
down = [(i, o, index < len(pairs) - 1) for index, (i, o) in enumerate(pairs)]
mid = [(down_dims[-1], down_dims[-1])] * 2
up = [(width_in * 2, width_out, True) for width_out, width_in in reversed(pairs[1:])]
return {"down": down, "mid": mid, "up": up}
```
For `down_dims = (16, 32, 64)` and `action_dim = 7`: `pairs = [(7, 16), (16, 32), (32, 64)]`; `up` is `[(128, 32, True), (64, 16, True)]`.

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 10B.1
from __future__ import annotations

def unet_channel_plan(action_dim: int, down_dims: Sequence[int]) -> dict[str, list[tuple]]:
    """Channel plan of the pinned conditional 1D U-Net.

    Returns ``{"down": [...], "mid": [...], "up": [...]}``. Each ``down`` entry is
    ``(in_channels, out_channels, downsample)``: two residual blocks ``in -> out`` and
    ``out -> out``, then a stride-2 convolution unless it is the last stage. ``mid`` is
    ``[(c, c), (c, c)]`` with ``c = down_dims[-1]``. Each ``up`` entry is
    ``(in_channels, out_channels, upsample)`` for the stages in reverse order, skipping the
    first ``(action_dim, down_dims[0])`` pair. Its first block takes ``2 * previous_width``
    channels because the matching encoder feature is concatenated. The pinned network
    upsamples after every ``up`` stage (the last included).
    """
    # Keep this function standalone when copied into notebook solution cells.
    pairs = [(action_dim, down_dims[0]), *zip(down_dims[:-1], down_dims[1:], strict=True)]  # noqa: RUF007
    down = [(i, o, index < len(pairs) - 1) for index, (i, o) in enumerate(pairs)]
    mid = [(down_dims[-1], down_dims[-1])] * 2
    up = [(width_in * 2, width_out, True) for width_out, width_in in reversed(pairs[1:])]
    return {"down": down, "mid": mid, "up": up}

In [ ]:
# Check: Exercise 10B.1
for dims in ((16, 32), (16, 32, 64), (8, 16, 24, 32)):
    pinned = pinned_unet(dims)
    plan = unet_channel_plan(7, dims)
    assert set(plan) == {"down", "mid", "up"}, "Return the keys down, mid and up."
    assert len(plan["down"]) == len(pinned.down_modules) and len(plan["up"]) == len(pinned.up_modules) and len(plan["mid"]) == len(pinned.mid_modules), f"Wrong number of stages for {dims}."
    for (i, o, down), (res1, res2, downsample) in zip(plan["down"], pinned.down_modules, strict=True):
        assert (res1.conv1.block[0].in_channels, res1.conv1.block[0].out_channels) == (i, o), f"down stage channels for {dims}: expected {(res1.conv1.block[0].in_channels, res1.conv1.block[0].out_channels)}, got {(i, o)}"
        assert res2.conv2.block[0].out_channels == o
        assert isinstance(downsample, nn.Conv1d) == bool(down), f"downsample flag for {dims}"
    for (c_in, c_out), block in zip(plan["mid"], pinned.mid_modules, strict=True):
        assert (block.conv1.block[0].in_channels, block.conv1.block[0].out_channels) == (c_in, c_out)
    for (i, o, up), (res1, res2, upsample) in zip(plan["up"], pinned.up_modules, strict=True):
        assert (res1.conv1.block[0].in_channels, res1.conv1.block[0].out_channels) == (i, o), f"up stage channels for {dims}: expected {(res1.conv1.block[0].in_channels, res1.conv1.block[0].out_channels)}, got {(i, o)}"
        assert isinstance(upsample, nn.ConvTranspose1d) == bool(up), f"upsample flag for {dims}"
print("Exercise 10B.1 passed")

**Use your function.** Build a U-Net from your plan and look at the sequence length and channel width along the way.

In [ ]:
# Use your function: Exercise 10B.1
unet = ConditionalUnet1D(7, 6, (16, 32, 64), step_embed_dim=32, plan_fn=unet_channel_plan)
print("plan:", unet_channel_plan(7, (16, 32, 64)))
print("parameters:", sum(p.numel() for p in unet.parameters()), "| pinned network with the same sizes:", sum(p.numel() for p in PINNED.parameters()))
x = torch.randn(1, 16, 7)
length = 16
for res1, res2, downsample in unet.down_blocks:
    length = length // 2 if isinstance(downsample, nn.Conv1d) else length
    print("after encoder stage: sequence length", length)

## Exercise 10B.2 — The forward pass with skip connections (about 15–20 min)

**Purpose.** Run the encoder, remember each stage's output, run the middle, then the decoder with the skips concatenated in reverse order.

**Contract (shapes, units, masks).**

- `unet` has `time_mlp` (timestep → vector), `down_blocks`, `mid_blocks`, `up_blocks`, `head`. `down_blocks` and `up_blocks` hold triples `(res1, res2, resample)`; `res1(x, cond)` and `res2(x, cond)` are residual blocks and `resample(x)` is a downsampling or upsampling module (or `Identity`).
- `x (B, H, A)` (**time before channels**), `timestep (B,)` long, `global_cond (B, G)` or `None`. Return `(B, H, A)`.
- Move to channels first: `(B, A, H)`. `cond = unet.time_mlp(timestep)`; if `global_cond` is not `None`, concatenate it on the last axis: `cond = [time vector, global_cond]`.
- Encoder: for each triple apply `res1`, then `res2`, **save that output as a skip feature**, then apply `resample`.
- Middle: apply each block of `mid_blocks` to `(x, cond)`.
- Decoder: for each triple, first concatenate `x` with the **most recently saved skip** on the channel axis (`dim=1`), then `res1`, `res2`, then `resample`. Skips are used in reverse order of saving.
- End with `head(x)` and move back to `(B, H, A)`.

**Method and source.** U-Net with skip connections ([Ronneberger et al. 2015](https://arxiv.org/abs/1505.04597), used for denoising in [DDPM](https://arxiv.org/abs/2006.11239) appendix B). Pinned code: `DiffusionConditionalUnet1d.forward`.

**Pointers.** Use a Python list as a stack: `append` after the second residual block of each encoder stage, `pop` in the decoder. Use `torch.cat((x, skip), dim=1)` for channels-first tensors.

<details><summary>Detailed hint</summary>

```python
x = x.transpose(1, 2)                                   # (B, A, H)
cond = unet.time_mlp(timestep)
if global_cond is not None:
    cond = torch.cat([cond, global_cond], dim=-1)
skips = []
for res1, res2, downsample in unet.down_blocks:
    x = res2(res1(x, cond), cond)
    skips.append(x)
    x = downsample(x)
for block in unet.mid_blocks:
    x = block(x, cond)
for res1, res2, upsample in unet.up_blocks:
    x = torch.cat((x, skips.pop()), dim=1)
    x = res2(res1(x, cond), cond)
    x = upsample(x)
return unet.head(x).transpose(1, 2)
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 10B.2
from __future__ import annotations

def unet_forward(
    unet: nn.Module, x: Tensor, timestep: Tensor, global_cond: Tensor | None = None
) -> Tensor:
    """Conditional U-Net forward pass with skip connections.

    ``x (B, H, A)`` (time before channels), ``timestep (B,)``, optional ``global_cond (B, G)``;
    returns ``(B, H, A)``. Steps: move to ``(B, A, H)``; ``cond = time_mlp(timestep)``, with
    ``global_cond`` concatenated on the last axis; for each ``down_blocks`` entry
    ``(res1, res2, downsample)`` apply ``res1(x, cond)``, ``res2(x, cond)``, store the result
    as a skip feature, then ``downsample``; run ``mid_blocks``; for each ``up_blocks`` entry
    ``(res1, res2, upsample)`` concatenate the newest stored skip on the channel axis, apply
    the two blocks, then ``upsample``; finish with ``head`` and return to ``(B, H, A)``.
    """
    x = x.transpose(1, 2)
    cond = unet.time_mlp(timestep)
    if global_cond is not None:
        cond = torch.cat([cond, global_cond], dim=-1)
    skips = []
    for res1, res2, downsample in unet.down_blocks:
        x = res2(res1(x, cond), cond)
        skips.append(x)
        x = downsample(x)
    for block in unet.mid_blocks:
        x = block(x, cond)
    for res1, res2, upsample in unet.up_blocks:
        x = torch.cat((x, skips.pop()), dim=1)
        x = res2(res1(x, cond), cond)
        x = upsample(x)
    return unet.head(x).transpose(1, 2)

In [ ]:
# Check: Exercise 10B.2
# The learner network has your forward function and the package plan; weights come from the pinned network.
learner = unet_from_lerobot(PINNED, forward_fn=unet_forward, plan_fn=ref.unet_channel_plan)
out = learner(X, T_STEPS, COND)
assert out.shape == X.shape, "Return (B, H, A)."
close(out, PINNED(X, T_STEPS, COND), "output differs from the pinned U-Net")
x1 = X.clone().requires_grad_(True); learner(x1, T_STEPS, COND).square().sum().backward()
x2 = X.clone().requires_grad_(True); PINNED(x2, T_STEPS, COND).square().sum().backward()
close(x1.grad, x2.grad, "input gradient", atol=1e-4)
close(learner.head[1].weight.grad, PINNED.final_conv[1].weight.grad, "head gradient", atol=1e-4)
close(learner.down_blocks[0][0].conv1.block[0].weight.grad, PINNED.down_modules[0][0].conv1.block[0].weight.grad, "first-block gradient (flows through the skip)", atol=1e-4)
close(learner(X, T_STEPS, COND), out, "repeatable")
no_cond = unet_from_lerobot(pinned_unet((16, 32), cond_dim=0), forward_fn=unet_forward, plan_fn=ref.unet_channel_plan)
close(no_cond(X, T_STEPS, None), pinned_unet((16, 32), cond_dim=0)(X, T_STEPS, None), "network without a global condition", atol=1e-5)
print("Exercise 10B.2 passed")

**Use your function.** Run your plan and your forward pass together in one network, with weights copied from the pinned one, and compare outputs.

In [ ]:
# Use your function: Exercise 10B.2
learner = unet_from_lerobot(PINNED, forward_fn=unet_forward, plan_fn=unet_channel_plan)
print("largest output difference from the pinned U-Net:", verify_unet_parity(learner, PINNED, X, T_STEPS, COND))

## Exercise 10B.3 — The weight converter (about 10 min)

**Purpose.** Move trained weights from the pinned network into your modules *explicitly*, so that a changed network cannot load by accident.

**Contract (shapes, units, masks).**

- `convert_lerobot_unet_state_dict(source)`: `source` maps tensor names to tensors (a LeRobot `state_dict`). Return a **new dict** for `ConditionalUnet1D`.
- Only the first name part changes: `diffusion_step_encoder → time_mlp`, `down_modules → down_blocks`, `mid_modules → mid_blocks`, `up_modules → up_blocks`, `final_conv → head`. Everything after the first dot is unchanged (for example `down_modules.0.0.conv1.block.0.weight` becomes `down_blocks.0.0.conv1.block.0.weight`).
- Every source tensor appears exactly once in the result (copy them).
- A name whose first part is not in the list must raise `KeyError`. Do not skip it silently.

**Method and source.** An explicit rename table with a loud failure is the safe way to move weights between two implementations; `load_state_dict(..., strict=True)` then catches missing or extra tensors on the other side.

**Pointers.** Split each name at the first dot with `partition('.')`. Look the first part up in a dict. Raise when it is missing.

<details><summary>Detailed hint</summary>

```python
renames = {"diffusion_step_encoder": "time_mlp", "down_modules": "down_blocks", "mid_modules": "mid_blocks",
           "up_modules": "up_blocks", "final_conv": "head"}
converted = {}
for name, value in source.items():
    top, _, rest = name.partition(".")
    if top not in renames:
        raise KeyError(f"no conversion rule for LeRobot U-Net tensor {name!r}")
    converted[f"{renames[top]}.{rest}"] = value.detach().clone()
return converted
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

In [ ]:
# Reference solution for Exercise 10B.3
from __future__ import annotations

def convert_lerobot_unet_state_dict(source: Mapping[str, Tensor]) -> dict[str, Tensor]:
    """Rename the tensors of the pinned ``DiffusionConditionalUnet1d`` for ``ConditionalUnet1D``.

    Only the first name part differs: ``diffusion_step_encoder`` -> ``time_mlp``,
    ``down_modules`` -> ``down_blocks``, ``mid_modules`` -> ``mid_blocks``,
    ``up_modules`` -> ``up_blocks``, ``final_conv`` -> ``head``. Everything after it is unchanged.
    An unknown first name raises ``KeyError`` so a changed network cannot slip through. Returns a
    new dict; every source tensor appears exactly once (copied).
    """
    renames = {
        "diffusion_step_encoder": "time_mlp",
        "down_modules": "down_blocks",
        "mid_modules": "mid_blocks",
        "up_modules": "up_blocks",
        "final_conv": "head",
    }
    converted: dict[str, Tensor] = {}
    for name, value in source.items():
        top, _, rest = name.partition(".")
        if top not in renames:
            raise KeyError(f"no conversion rule for LeRobot U-Net tensor {name!r}")
        converted[f"{renames[top]}.{rest}"] = value.detach().clone()
    return converted

In [ ]:
# Check: Exercise 10B.3
converted = convert_lerobot_unet_state_dict(PINNED.state_dict())
assert len(converted) == len(PINNED.state_dict()), "Every source tensor must appear exactly once."
fresh = ConditionalUnet1D(7, 6, (16, 32, 64), step_embed_dim=32)
assert set(converted) == set(fresh.state_dict()), f"Names differ from the learner module: {sorted(set(converted) ^ set(fresh.state_dict()))[:4]}"
for name, tensor in converted.items():
    assert tensor.shape == fresh.state_dict()[name].shape, f"shape of {name}"
fresh.load_state_dict(converted, strict=True)
close(fresh.eval()(X, T_STEPS, COND), PINNED(X, T_STEPS, COND), "converted weights must reproduce the pinned output")
try:
    convert_lerobot_unet_state_dict({"mystery.weight": torch.zeros(1)})
except KeyError:
    pass
else:
    raise AssertionError("An unknown tensor name must raise KeyError.")
source_key = next(iter(PINNED.state_dict()))
assert converted[convert_lerobot_unet_state_dict({source_key: torch.zeros(1)}).popitem()[0]] is not PINNED.state_dict()[source_key], "Copy the tensors."
print("Exercise 10B.3 passed")

**Use your function.** Load the *prepared* checkpoint's U-Net into your modules through your converter, your plan and your forward function. Check numeric parity on real camera frames before using the network anywhere.

In [ ]:
# Use your function: Exercise 10B.3
RUN_PREPARED_UNET_PARITY = True

if RUN_PREPARED_UNET_PARITY:
    import copy
    from torch.utils.data import default_collate
    from vla_lab.teaching.act_model import open_libero_dataset
    from vla_lab.teaching.diffusion import load_prepared_diffusion, stack_observation
    from vla_lab.training.diffusion import FPS, diffusion_delta_timestamps

    checkpoint = DATA / "runs/diffusion-bowl-plate-reference-2000/checkpoint-002000"
    if not (checkpoint / "model.safetensors").is_file():
        print("Prepared diffusion checkpoint not found; skipping this check.")
    else:
        policy, preprocessor, _ = load_prepared_diffusion(checkpoint, device="cpu")
        prepared_unet = policy.diffusion.unet
        yours = unet_from_lerobot(prepared_unet, converter=convert_lerobot_unet_state_dict, plan_fn=unet_channel_plan,
                                  forward_fn=unet_forward, block_forward_fn=film_residual_forward, embedding_fn=sinusoidal_timestep_embedding)
        dataset = open_libero_dataset(CONFIGS, DATA, "bowl_plate", "validation", diffusion_delta_timestamps(policy.config, FPS))
        batch = preprocessor(default_collate([dataset[i] for i in (5, 60)]))
        with torch.no_grad():
            cond = policy.diffusion._prepare_global_conditioning(stack_observation(policy, batch))   # real frames -> condition
            noisy = torch.randn(2, 16, 7, generator=torch.Generator().manual_seed(0))
            difference = verify_unet_parity(yours, prepared_unet, noisy, torch.tensor([7, 77]), cond)
        print("real-size U-Net, real camera condition: largest output difference", difference)
        assert difference < 1e-4, "Reference weights in your U-Net must reproduce the pinned network before any experiment."
else:
    print("Prepared-U-Net parity is off.")

## Opt-in: update your U-Net inside a real policy

This opt-in check loads the prepared diffusion policy and copies its U-Net weights into your U-Net. It checks real-condition parity, then runs two updates with the Unit 9 `epsilon_loss_fn` through the Unit 9 training loop. The image encoder stays frozen. Only local files are used. The update budget is two; this is an integration check, not a training result. Set `RUN_REAL_UNET_UPDATE = True` to run it.

In [ ]:
RUN_REAL_UNET_UPDATE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REAL_UPDATES = 2

if RUN_REAL_UNET_UPDATE:
    from torch.utils.data import default_collate
    from vla_lab.training.diffusion import FPS, diffusion_delta_timestamps

    checkpoint = DATA / "runs/diffusion-bowl-plate-reference-2000/checkpoint-002000"
    if not (checkpoint / "model.safetensors").is_file():
        raise FileNotFoundError(f"Prepared diffusion checkpoint missing: {checkpoint}")
    policy, preprocessor, _ = load_prepared_diffusion(checkpoint, device=DEVICE)
    policy.eval()
    for parameter in policy.parameters():
        parameter.requires_grad_(False)
    pinned_unet = policy.diffusion.unet
    learner_unet = unet_from_lerobot(
        pinned_unet, converter=convert_lerobot_unet_state_dict, plan_fn=unet_channel_plan,
        forward_fn=unet_forward, block_forward_fn=film_residual_forward,
        embedding_fn=sinusoidal_timestep_embedding,
    ).to(DEVICE)
    dataset = open_libero_dataset(CONFIGS, DATA, "bowl_plate", "train", diffusion_delta_timestamps(policy.config, FPS), episodes=[379, 422])
    batch = preprocessor(default_collate([dataset[i] for i in (3, 40)]))
    class FrozenPolicyDenoiser(LeRobotDenoiser):
        def train(self, mode=True):
            super().train(mode)
            self.policy.eval()
            self.unet.train(mode)
            return self

    denoiser = FrozenPolicyDenoiser(policy, unet=learner_unet)
    noisy = torch.randn(2, policy.config.horizon, policy.config.action_feature.shape[0], device=DEVICE,
                        generator=torch.Generator(device=DEVICE).manual_seed(3))
    timestep = torch.tensor([7, 77], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        condition = denoiser.global_condition(batch)
        difference = verify_unet_parity(learner_unet, pinned_unet, noisy, timestep, condition)
    print("real-condition parity difference:", difference)
    assert difference < 1e-4, "Your U-Net must match the pinned network before training."
    optimizer = torch.optim.AdamW(learner_unet.parameters(), lr=1e-5)
    before = learner_unet.head[1].weight.detach().clone()
    torch.manual_seed(4)
    history = train_loop(denoiser, [batch], optimizer, epsilon_loss_fn, steps=REAL_UPDATES)
    changed = not torch.equal(before, learner_unet.head[1].weight)
    print("updates:", [round(row["loss"], 5) for row in history], "| learner U-Net changed:", changed)
    assert changed, "The real training path must update your U-Net weights."
    assert all(parameter.grad is None for parameter in policy.parameters()), "The pinned policy weights must stay frozen."
else:
    print("Real U-Net updates are off. Set RUN_REAL_UNET_UPDATE=True to run the two-update check.")

## Experiment — change the architecture under a fixed budget (30–60 min)

**Question.** With the same number of updates, how do depth and kernel size of the U-Net change the held-out noise-prediction error and the cost?

**Gate first.** The harness refuses to run any trial unless your U-Net (your plan, forward pass and converter) reproduces the pinned network to `1e-4` on the algorithm-check inputs. An experiment on a network you have not checked is not an experiment.

**Setup (real data).** Real windows as in Unit 9 (state-conditioned, no images), cosine schedule, your Unit 10 functions in the real network, the Unit 9 loss callback in `train_loop`.

**Fixed for every trial.** Weight seed `0`; batch seed `1`; **200 updates**, batch 64, Adam(`lr=1e-3`, betas `(0.95, 0.999)`); the same validation windows and noise (seed `7`).

**Allowed changes (per trial).** `dims`: `shallow` = `(32, 64)`, `base` = `(32, 64, 128)`, `deep` = `(16, 32, 64, 128)`. `kernel`: `3` or `5`. (The horizon 16 is divisible by `2^4`, so four stages are valid.)

**Measures.** `eps_grid`: noise-prediction MSE on validation windows (real steps), averaged over timesteps `10, 30, 50, 70, 90`, fixed noise. `parameters`. `ms_per_update`: wall-clock milliseconds per training update on this CPU (noisy; compare only within one session).

**Prediction first.** In the next cell: which architecture will have the lowest `eps_grid`, and will it be the biggest one?

**Keep failed trials.** Every trial is appended to `TRIALS`; an architecture that is too deep for the horizon is a failed trial worth keeping.

In [ ]:
# Larger training experiment: run only when you opt in.
RUN_EXPERIMENT = False

In [ ]:
if RUN_EXPERIMENT:
    PREDICTION = "TODO: replace this text with your prediction."
    print(PREDICTION)

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: trial table helper
    import time
    import pandas as pd

    def show_trials(trials, columns):
        """Print every trial that was run, including failed ones, and return a table."""
        table = pd.DataFrame([{**t["config"], **{c: t.get(c) for c in columns}, "status": t["status"]} for t in trials])
        print(table.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
        return table

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: the parity gate and the real windows
    gate = unet_from_lerobot(PINNED, converter=convert_lerobot_unet_state_dict, plan_fn=unet_channel_plan,
                             forward_fn=unet_forward, block_forward_fn=film_residual_forward, embedding_fn=sinusoidal_timestep_embedding)
    gate_difference = verify_unet_parity(gate, PINNED, X, T_STEPS, COND)
    assert gate_difference < 1e-4, f"Your U-Net differs from the pinned network by {gate_difference:.1e}; fix the exercises before the experiment."
    print("parity gate passed:", gate_difference)

    TRAIN = load_state_action_windows(CONFIGS, DATA, "bowl_plate", "train")
    VALID = load_state_action_windows(CONFIGS, DATA, "bowl_plate", "validation")

In [ ]:
if RUN_EXPERIMENT:
    # Supplied: harness. Your plan, forward pass and Unit 10A blocks build and run the network.
    STEPS, BATCH, LR, SEED_WEIGHTS, SEED_BATCH, SEED_NOISE = 200, 64, 1e-3, 0, 1, 7
    SCHEDULE, GRID = DDPMSchedule.cosine(100), (10, 30, 50, 70, 90)
    DIMS = {"shallow": (32, 64), "base": (32, 64, 128), "deep": (16, 32, 64, 128)}
    BASELINE = {"dims": "base", "kernel": 5}
    TRIALS = []


    def run_trial(config):
        torch.manual_seed(SEED_WEIGHTS)
        unet = ConditionalUnet1D(
            7, 16, DIMS[config["dims"]], kernel_size=config["kernel"], step_embed_dim=32,
            plan_fn=unet_channel_plan, forward_fn=unet_forward,
            block_forward_fn=film_residual_forward, embedding_fn=sinusoidal_timestep_embedding,
        )
        model = DiffusionDenoiser(unet, SCHEDULE, flat_state_condition)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.95, 0.999))
        draws = torch.Generator().manual_seed(SEED_BATCH)
        batches = [take_batch(TRAIN, torch.randint(0, len(TRAIN["action"]), (BATCH,), generator=draws)) for _ in range(STEPS)]
        started = time.perf_counter()
        train_loop(model, batches, optimizer, epsilon_loss_fn, steps=STEPS)
        seconds = time.perf_counter() - started
        model.eval()
        with torch.no_grad():
            noise = torch.randn(VALID["action"].shape, generator=torch.Generator().manual_seed(SEED_NOISE))
            real = ~VALID["action_is_pad"]
            own = []
            for t in GRID:
                times = torch.full((len(noise),), t)
                noisy = forward_noise(VALID["action"], noise, times, SCHEDULE.alpha_bar)
                own.append((model(noisy, times, VALID) - noise).square().mean(-1)[real].mean().item())
        return {"eps_grid": sum(own) / len(own), "parameters": sum(p.numel() for p in model.parameters()),
                "ms_per_update": 1000 * seconds / STEPS}


    def record(config):
        config = {**BASELINE, **config}
        try:
            TRIALS.append({"config": config, "status": "ok", **run_trial(config)})
        except Exception as error:  # a failed trial is evidence too
            TRIALS.append({"config": config, "status": f"failed: {type(error).__name__}: {error}"})


    record(BASELINE)
    COLUMNS = ["eps_grid", "parameters", "ms_per_update"]
    show_trials(TRIALS, COLUMNS);

**Your trials.** Add at least two trials that each change one setting. Budget: 200 updates per trial.

In [ ]:
if RUN_EXPERIMENT:
    # Reference choice of trials: one change per trial.
    MY_TRIALS = [{"dims": "shallow"}, {"dims": "deep"}, {"kernel": 3}]
    for trial in MY_TRIALS:
        record(trial)
    show_trials(TRIALS, COLUMNS);

In [ ]:
if RUN_EXPERIMENT:
    # Check: the protocol was followed.
    assert TRIALS[0]["config"] == BASELINE, "The first trial must be the baseline."
    assert len(TRIALS) >= 3, "Run the baseline and at least two trials of your own."
    assert all(set(t["config"]) == set(BASELINE) for t in TRIALS), "Only dims and kernel may change."
    assert all(t["config"]["dims"] in DIMS and t["config"]["kernel"] in (3, 5) for t in TRIALS)
    ok = [t for t in TRIALS if t["status"] == "ok"]
    assert all(torch.isfinite(torch.tensor(t["eps_grid"])) for t in ok), "eps_grid must be finite."
    print(f"{len(TRIALS)} trials kept ({len(ok)} ok). Budget: {STEPS} updates, batch {BATCH}, seeds {SEED_WEIGHTS}/{SEED_BATCH}/{SEED_NOISE}.")

In [ ]:
if RUN_EXPERIMENT:
    # Use your function: quality against cost, every trial shown
    ok = [t for t in TRIALS if t["status"] == "ok"]
    labels = [", ".join(f"{k}={v}" for k, v in t["config"].items() if v != BASELINE[k]) or "baseline" for t in ok]
    fig, axis = plt.subplots(figsize=(5.5, 4))
    axis.scatter([t["parameters"] for t in ok], [t["eps_grid"] for t in ok])
    for t, label in zip(ok, labels):
        axis.annotate(label, (t["parameters"], t["eps_grid"]), fontsize=8)
    axis.set_xlabel("parameters"); axis.set_ylabel("validation eps_grid"); axis.grid(alpha=0.3); plt.show()

**Explain (2–4 sentences, edit this cell).** Which architecture was best at this budget, and is the largest network the best one? Compare quality with parameters and time. What do the failed trials, if any, tell you about the 16-step horizon? Why can you not conclude that a deeper network is better for the *robot*?

*Optional extension:* train the baseline longer (change `STEPS` for every trial and keep the old rows) to see whether the ranking changes.

## What you built

- `unet_channel_plan`, `unet_forward` and `convert_lerobot_unet_state_dict`: a conditional U-Net that loads the pinned weights and reproduces the pinned outputs on random inputs and on real camera conditions.
- A fixed-budget architecture comparison gated by numeric parity, with every trial kept.

**Next:** Unit 11 runs the network backwards: DDPM sampling, skipped timesteps, and the actions that are actually executed.